# 01: Initial EDA on HMDA Snapshot LAR (2022 to 2024)

Milestone 1 deliverable. Purpose: establish baseline understanding of the raw HMDA snapshot files before writing any dbt model.

Scope of this notebook:
1. Verify local data presence and read the manifest.
2. Row counts per activity year.
3. Null pattern survey for the fields that will anchor the staging layer.
4. Distribution snapshots: action_taken, loan_type, loan_purpose, lender size proxies.
5. First-pass lender concentration check.
6. Summary of findings to carry into /docs/data-quality-notes.md.

This notebook is intentionally read-mostly. No writes to the warehouse, no derived tables persisted. DuckDB scans the raw CSV files in place.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'CLAUDE.md').exists():
    REPO_ROOT = REPO_ROOT.parent
print(f'Repo root: {REPO_ROOT}')

HMDA_DIR = REPO_ROOT / 'data' / 'raw' / 'hmda'
MANIFEST = HMDA_DIR / '_manifest.json'

## 1. Data presence and manifest

In [ ]:
if not MANIFEST.exists():
    print('HMDA manifest not found. Run:')
    print('    python scripts/download_hmda.py --years 2022 2023 2024')
    print('then re-run this notebook.')
else:
    manifest = json.loads(MANIFEST.read_text())
    df_m = pd.DataFrame(manifest.get('entries', []))
    print(f'{len(df_m)} manifest entries')
    if not df_m.empty:
        display(df_m[['year', 'asset', 'bytes', 'sha256', 'downloaded_at_utc']])

## 2. Row counts per year

DuckDB reads zipped CSVs directly via the httpfs / zip scan path. For very large files, this is faster and more memory-friendly than pandas.

Expected ballpark: 15M to 25M rows per year depending on rate environment. The 2022 to 2024 window spans the post-COVID rate spike, so volume should drop materially from 2022 to 2023.

In [ ]:
con = duckdb.connect()

def lar_csv_path(year: int) -> Path:
    return HMDA_DIR / str(year) / f'{year}_public_lar_nationwide.csv'

row_counts = []
for year in (2022, 2023, 2024):
    p = lar_csv_path(year)
    if not p.exists():
        print(f'{year}: missing ({p.relative_to(REPO_ROOT)})')
        continue
    q = f"SELECT COUNT(*) AS rows FROM read_csv_auto('{p}', compression='zstd', sample_size=-1)"
    try:
        rows = con.execute(q).fetchone()[0]
    except Exception:
        q = f"SELECT COUNT(*) AS rows FROM read_csv_auto('{p}')"
        rows = con.execute(q).fetchone()[0]
    row_counts.append({'activity_year': year, 'rows': rows})

df_rows = pd.DataFrame(row_counts)
display(df_rows)

## 3. Null pattern survey on anchor fields

Fields examined because they anchor the staging layer contract:

- `lei` (post-2018 lender identifier)
- `activity_year`
- `loan_application_number` (composes the PK with LEI and year)
- `state_code`, `county_code`, `census_tract` (geography join)
- `action_taken` (the dependent variable for most analyses)
- `loan_type`, `loan_purpose`, `occupancy_type` (portfolio composition)
- `loan_amount`, `income`, `property_value`
- Demographic fields (`derived_race`, `derived_ethnicity`, `derived_sex`)

'Exempt' values in numeric fields appear as literal strings; counted separately.

In [ ]:
ANCHOR_FIELDS = [
    'lei', 'activity_year', 'loan_application_number',
    'state_code', 'county_code', 'census_tract',
    'action_taken', 'loan_type', 'loan_purpose', 'occupancy_type',
    'loan_amount', 'income', 'property_value',
    'derived_race', 'derived_ethnicity', 'derived_sex',
    'rate_spread', 'interest_rate', 'debt_to_income_ratio',
]

def null_and_exempt_profile(year: int) -> pd.DataFrame:
    p = lar_csv_path(year)
    if not p.exists():
        return pd.DataFrame()
    selects = []
    for f in ANCHOR_FIELDS:
        selects.append(
            f"SUM(CASE WHEN {f} IS NULL THEN 1 ELSE 0 END) AS {f}__null, "
            f"SUM(CASE WHEN CAST({f} AS VARCHAR) = 'Exempt' THEN 1 ELSE 0 END) AS {f}__exempt"
        )
    q = f"SELECT {', '.join(selects)} FROM read_csv_auto('{p}')"
    try:
        row = con.execute(q).fetchdf().iloc[0]
    except Exception as e:
        print(f'{year}: column surface differs from expected. {e}')
        return pd.DataFrame()
    long = []
    for f in ANCHOR_FIELDS:
        long.append({
            'field': f,
            'null': int(row.get(f'{f}__null', 0)),
            'exempt': int(row.get(f'{f}__exempt', 0)),
        })
    return pd.DataFrame(long).assign(year=year)

parts = []
for y in (2022, 2023, 2024):
    p = null_and_exempt_profile(y)
    if not p.empty:
        parts.append(p)

if parts:
    df_null = pd.concat(parts, ignore_index=True)
    display(df_null.pivot_table(index='field', columns='year', values=['null', 'exempt'], fill_value=0))

## 4. Distribution snapshots

Goal: eyeball whether the distributions look anywhere close to what CFPB Data Browser aggregates would show for the same vintage. Anomalies flagged here get written to /docs/data-quality-notes.md.

In [ ]:
def action_taken_mix(year: int) -> pd.DataFrame:
    p = lar_csv_path(year)
    if not p.exists():
        return pd.DataFrame()
    q = f"""
        SELECT action_taken, COUNT(*) AS rows
        FROM read_csv_auto('{p}')
        GROUP BY 1 ORDER BY 1
    """
    df = con.execute(q).fetchdf()
    df['year'] = year
    df['share'] = df['rows'] / df['rows'].sum()
    return df

mix = pd.concat([action_taken_mix(y) for y in (2022, 2023, 2024)], ignore_index=True)
if not mix.empty:
    display(mix.pivot_table(index='action_taken', columns='year', values='share', fill_value=0).round(4))

In [ ]:
def loan_type_mix(year: int) -> pd.DataFrame:
    p = lar_csv_path(year)
    if not p.exists():
        return pd.DataFrame()
    q = f"""
        SELECT loan_type, loan_purpose, COUNT(*) AS rows
        FROM read_csv_auto('{p}')
        WHERE action_taken = 1
        GROUP BY 1,2 ORDER BY 1,2
    """
    df = con.execute(q).fetchdf()
    df['year'] = year
    return df

orig_mix = pd.concat([loan_type_mix(y) for y in (2022, 2023, 2024)], ignore_index=True)
if not orig_mix.empty:
    display(orig_mix.pivot_table(index=['loan_type', 'loan_purpose'], columns='year', values='rows', fill_value=0))

## 5. Lender concentration and panel size

How many distinct LEIs report each year, and what share of originations does the top 10 take?

In [ ]:
def lender_concentration(year: int) -> dict:
    p = lar_csv_path(year)
    if not p.exists():
        return {}
    q = f"""
        WITH orig AS (
            SELECT lei, COUNT(*) AS originations
            FROM read_csv_auto('{p}')
            WHERE action_taken = 1
            GROUP BY 1
        ),
        ranked AS (
            SELECT lei, originations, ROW_NUMBER() OVER (ORDER BY originations DESC) AS rnk
            FROM orig
        )
        SELECT
            (SELECT COUNT(*) FROM ranked) AS distinct_lenders,
            (SELECT SUM(originations) FROM ranked) AS total_originations,
            (SELECT SUM(originations) FROM ranked WHERE rnk <= 10) AS top10_originations
    """
    row = con.execute(q).fetchone()
    return {
        'year': year,
        'distinct_lenders': row[0],
        'total_originations': row[1],
        'top10_originations': row[2],
        'top10_share': (row[2] / row[1]) if row[1] else None,
    }

conc = pd.DataFrame([lender_concentration(y) for y in (2022, 2023, 2024)])
if not conc.empty:
    display(conc)

## 6. Stock Yards Bank and Trust: presence check

Sanity check that the anchor reporter used in the case study appears in the data. SYB files HMDA under a stable LEI. If this cell returns no rows, investigate before proceeding: either the LEI changed, a merger event has to be reconciled, or the raw file is incomplete.

In [ ]:
SYB_LEI_CANDIDATES = [
    '4LJGQ9KJ9S0CP4B1FY29',
]

def syb_presence(year: int) -> pd.DataFrame:
    p = lar_csv_path(year)
    if not p.exists():
        return pd.DataFrame()
    lei_list = "', '".join(SYB_LEI_CANDIDATES)
    q = f"""
        SELECT lei, action_taken, COUNT(*) AS rows
        FROM read_csv_auto('{p}')
        WHERE lei IN ('{lei_list}')
        GROUP BY 1, 2 ORDER BY 1, 2
    """
    df = con.execute(q).fetchdf()
    df['year'] = year
    return df

syb = pd.concat([syb_presence(y) for y in (2022, 2023, 2024)], ignore_index=True)
if syb.empty:
    print('No rows found for candidate LEIs. Verify LEI against GLEIF before concluding absence.')
else:
    display(syb)

## 7. Observations to carry forward

To be populated after a first full run against actual data. Target destination: /docs/data-quality-notes.md.

Checklist of patterns to watch for:
- Exempt share by field (which lenders are partial-exemption reporters, how much data is lost)
- Null share on anchor fields (anything unexpected above single-digit percent)
- action_taken code 6 (purchased loans) share and whether it concentrates among specific LEIs
- rate_spread and interest_rate reporting completeness for originated loans
- Any LEI present in one year but not adjacent years (merger, charter change, reporting threshold crossing)
- Geographic coverage holes by state